# 06 — Attention from scratch

The lecture built a transformer as small, inspectable parts. This
notebook isolates the routing step: **compare queries with keys,
normalize over keys, then mix values**.

A catalog decoder predicts one token at a time:

$$
p(t_{1:T}\mid F,\theta)=\prod_i p(t_i\mid t_{<i},F,\theta).
$$

Causal self-attention reads only the catalog prefix. Cross-attention
reads the encoded field. Here “causal” describes token order, not
physical time.

In [1]:
from pathlib import Path
import json
import os

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path(os.path.abspath('.')).parent

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".mplconfig"))

SEED = 2603
COLORS = {
    "blue": "#2D6A9F",
    "orange": "#E6862E",
    "green": "#3A8D72",
    "purple": "#7656A5",
    "red": "#C94C4C",
    "gray": "#626C78",
}

def savefig(fig, name, evidence="analytic-fixture"):
    fig.text(0.995, 0.005, evidence, ha="right", va="bottom",
             fontsize=7, color=COLORS["gray"])
    path = OUTPUT_DIR / name
    fig.savefig(path, dpi=160, bbox_inches="tight", facecolor="white")
    print("saved:", path.relative_to(ROOT))


import torch
import torch.nn.functional as F

torch.manual_seed(SEED)
torch.set_num_threads(1)
rng = np.random.default_rng(SEED)

## The calculation

$$
A=\operatorname{softmax}_{T_k}
\left(\frac{QK^\top}{\sqrt{d_k}}+M\right),
\qquad O=AV.
$$

$A$ has shape $(B,H,T_q,T_k)$. Each query row is a probability
distribution over keys. A causal mask removes future catalog keys;
field cross-attention normally has no causal mask. The arrays below
are already projected Q, K, and V. A full multi-head layer also
learns the Q/K/V projections and a projection after the heads merge.
Every query must retain at least one allowed key. The output feature
width comes from V, not from Q or K.

In [2]:
def softmax(logits):
    '''Stable softmax over the key axis.'''
    # TODO: subtract the row maximum, exponentiate, and normalize.
    raise NotImplementedError


def attention(q, k, v, allowed=None, scale=True):
    '''Scaled dot-product attention for arrays ending in (token, feature).'''
    scores = q @ np.swapaxes(k, -2, -1)
    # TODO: apply the sqrt(d_k) scale, mask forbidden scores, then return
    # the weighted values and attention weights.
    raise NotImplementedError


def causal_allowed(length):
    '''True where a query may read a key.'''
    # TODO: return a lower-triangular Boolean array, including the diagonal.
    raise NotImplementedError


def split_heads(x, n_heads):
    '''(B,T,D) -> (B,H,T,D/H).'''
    # TODO: reshape to expose the head axis, then move heads before tokens.
    raise NotImplementedError


def merge_heads(x):
    '''(B,H,T,D/H) -> (B,T,D).'''
    # TODO: exactly invert split_heads.
    raise NotImplementedError

In [ ]:
# Self-attention has the same number of queries and keys.
T, width = 6, 8
q = rng.normal(size=(1, 2, T, width))
k = rng.normal(size=(1, 2, T, width))
v = rng.normal(size=(1, 2, T, 3))
causal = causal_allowed(T)
self_out, self_weights = attention(q, k, v, causal)

# Cross-attention can be rectangular: four catalog queries, six field patches.
q_cross = rng.normal(size=(1, 2, 4, width))
k_cross = rng.normal(size=(1, 2, 6, width))
v_cross = rng.normal(size=(1, 2, 6, 3))
cross_out, cross_weights = attention(q_cross, k_cross, v_cross)

# Multi-head attention is only a reshape plus a transpose around this calculation.
sentinel = np.arange(2 * 5 * 12).reshape(2, 5, 12)
round_trip = merge_heads(split_heads(sentinel, 3))
np.testing.assert_array_equal(round_trip, sentinel)

# With zero Q/K, causal attention returns the running mean of visible values.
simple_v = np.arange(T, dtype=float).reshape(1, 1, T, 1)
prefix_mean, _ = attention(
    np.zeros((1, 1, T, width)),
    np.zeros((1, 1, T, width)),
    simple_v,
    causal,
)
np.testing.assert_allclose(prefix_mean.ravel(), np.cumsum(np.arange(T)) / np.arange(1, T + 1))

# Positionless, unmasked self-attention is permutation equivariant.
permutation = np.array([3, 0, 5, 1, 4, 2])
plain_out, _ = attention(q, k, v)
permuted_out, _ = attention(
    q[:, :, permutation],
    k[:, :, permutation],
    v[:, :, permutation],
)
np.testing.assert_allclose(permuted_out, plain_out[:, :, permutation])

# One independent library comparison checks the complete attention calculation.
torch_out = F.scaled_dot_product_attention(
    torch.tensor(q, dtype=torch.float64),
    torch.tensor(k, dtype=torch.float64),
    torch.tensor(v, dtype=torch.float64),
    attn_mask=torch.tensor(causal[None, None]),
    dropout_p=0.0,
)
np.testing.assert_allclose(self_out, torch_out.numpy(), atol=2e-12, rtol=2e-12)

fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.5), constrained_layout=True)
for ax, matrix, title in [
    (axes[0], self_weights[0, 0], "causal self-attention"),
    (axes[1], cross_weights[0, 0], "field cross-attention"),
]:
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1, aspect="auto")
    ax.set(title=title, xlabel="key", ylabel="query")
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_yticks(range(matrix.shape[0]))
    ax.grid(False)
fig.colorbar(image, ax=axes, label="attention weight")
savefig(fig, "06_attention_mechanics.png")
plt.show()

print("self weights:", self_weights.shape, "| cross weights:", cross_weights.shape)
print("largest forbidden causal weight:", np.triu(self_weights[0, 0], 1).max())
print("split/merge identity and positionless permutation test: PASS")

## Why the scale and mask matter

Here Q and K contain independent standard-normal draws. Entropy is
measured over 12 key weights in nats, with uniform maximum
$\log 12\approx2.485$. Scaling keeps its distribution roughly
independent of head width; without it wider heads give sharper
softmaxes. Unmasked future mass naturally falls for later queries
because fewer future keys remain. Exactly zero masked mass and the
suffix-invariance test are the decisive leakage checks.

In [ ]:
widths = np.array([2, 4, 8, 16, 32, 64, 128])
entropy_scaled, entropy_unscaled = [], []
for width in widths:
    q0 = rng.normal(size=(256, 1, 1, width))
    k0 = rng.normal(size=(256, 1, 12, width))
    scores = q0 @ np.swapaxes(k0, -2, -1)
    for target, logits in [
        (entropy_scaled, scores / np.sqrt(width)),
        (entropy_unscaled, scores),
    ]:
        probability = softmax(logits)
        target.append(float(np.mean(-np.sum(probability * np.log(probability + 1e-15), axis=-1))))

_, unmasked_weights = attention(q, k, v)
future_mass = np.triu(unmasked_weights[0, 0], 1).sum(axis=1)
causal_future_mass = np.triu(self_weights[0, 0], 1).sum(axis=1)

# A direct leakage test: changing the whole future suffix must not alter earlier outputs.
q_changed, k_changed, v_changed = q.copy(), k.copy(), v.copy()
q_changed[:, :, 3:] += 100
k_changed[:, :, 3:] += 100
v_changed[:, :, 3:] += 100
changed_out, _ = attention(q_changed, k_changed, v_changed, causal)
np.testing.assert_allclose(changed_out[:, :, :3], self_out[:, :, :3])

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.5), constrained_layout=True)
axes[0].plot(widths, entropy_scaled, "o-", label=r"with $1/\sqrt{d_k}$")
axes[0].plot(widths, entropy_unscaled, "s--", label="without scale")
axes[0].set(xscale="log", xlabel=r"head width $d_k$", ylabel="entropy [nats]",
            title="Scaling stabilizes the softmax")
axes[0].legend()
axes[1].plot(future_mass, "o-", label="no mask")
axes[1].plot(causal_future_mass, "s--", label="causal mask")
axes[1].set(xlabel="query token", ylabel="future probability mass",
            title="The mask removes future information")
axes[1].legend()
savefig(fig, "06_scaling_and_leakage.png")
plt.show()

## Takeaway

- Self-attention reads the generated prefix.
- Cross-attention connects each catalog query to the field.
- Positions must be supplied separately; without them, unmasked
  self-attention is permutation equivariant.
- Attention weights show routing inside the network; they are not
  by themselves a causal explanation of the physics.

## After the core exercise

Make one change at a time and predict the result before running it:

1. Extend the head-width scan or remove $1/\sqrt{d_k}$; compare the
   attention entropy, not just the largest weight.
2. Reverse one triangle in `causal_allowed` and check whether the
   suffix-invariance test catches the leak.
3. Jointly permute cross-attention K and V. The output for a fixed Q
   should not change; permuting K alone should change it.
4. Add distinct position vectors to otherwise identical tokens and
   test how the positionless permutation result changes.